# Task 1.4 — Load SLA Contracts XLSX and Aircraft Master CSV

- SLA XLSX: use openpyxl/pandas to parse; write to `bronze.airline_sla` Delta table
- Validate: every `airline_code` in flight_schedule must have an SLA contract — log gaps
- Aircraft master: MERGE INTO on `aircraft_registration` (weekly refresh may update `age_years`)
- Cross-validation: every `aircraft_registration` in flight schedule must exist in aircraft_master


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import pandas as pd

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_1_4_SLA_Aircraft_Master_Bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Part A — Airline SLA Contracts Raw Ingestion

In [ ]:
# ── Raw Ingestion: airline_sla_contracts ──────────────────────────
# Try XLSX first (openpyxl), fall back to CSV
try:
    sla_pandas_df = pd.read_excel(
        f"{DATA_DIR}/airline_sla/airline_sla_contracts.xlsx",
        engine="openpyxl"
    )
    print("Read SLA from XLSX")
except Exception:
    sla_pandas_df = pd.read_csv(
        f"{DATA_DIR}/airline_sla/airline_sla_contracts.csv"
    )
    print("Read SLA from CSV fallback")

raw_sla_df = spark.createDataFrame(sla_pandas_df)
raw_sla_df = raw_sla_df.withColumn("ingestion_ts", F.current_timestamp())

print(f"Raw SLA records: {raw_sla_df.count()}")
raw_sla_df.printSchema()
raw_sla_df.show(10, truncate=False)


## Part A — SLA Data Quality Checks

In [ ]:
# ── DQ Checks: SLA Contracts ──────────────────────────────────────
# 1. No duplicate airline_codes
dup_airlines = raw_sla_df.groupBy("airline_code").count().filter("count > 1")
print(f"Duplicate airline_codes: {dup_airlines.count()}")
if dup_airlines.count() > 0:
    dup_airlines.show()

# 2. All time values must be positive integers
time_cols = [
    "min_turn_time_narrow_body_mins", "min_turn_time_wide_body_mins",
    "connection_time_domestic_mins", "connection_time_international_mins"
]
for col_name in time_cols:
    neg_vals = raw_sla_df.filter(F.col(col_name) <= 0)
    print(f"  {col_name} <= 0: {neg_vals.count()}")

# 3. otp_target_pct should be between 0 and 100
bad_otp = raw_sla_df.filter(
    (F.col("otp_target_pct") <= 0) | (F.col("otp_target_pct") > 100)
)
print(f"Invalid otp_target_pct: {bad_otp.count()}")

validated_sla_df = raw_sla_df  # All pass → keep as-is
print(f"\nValidated SLA records: {validated_sla_df.count()}")


## Part A — Write SLA to Bronze Delta

In [ ]:
# ── Write SLA Bronze Delta ────────────────────────────────────────
bronze_sla_path = f"{BRONZE_DIR}/airline_sla"

(validated_sla_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_sla_path))

print(f"SLA written to {bronze_sla_path}")
bronze_sla_df = spark.read.format("delta").load(bronze_sla_path)
print(f"Bronze airline_sla total rows: {bronze_sla_df.count()}")
bronze_sla_df.show(5, truncate=False)


## Part B — Aircraft Master Raw Ingestion

In [ ]:
# ── Raw Ingestion: aircraft_master.csv ────────────────────────────
raw_aircraft_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_DIR}/aircraft_master/aircraft_master.csv"))

raw_aircraft_df = raw_aircraft_df.withColumn(
    "ingestion_ts", F.current_timestamp()
)

print(f"Raw aircraft master records: {raw_aircraft_df.count()}")
raw_aircraft_df.printSchema()
raw_aircraft_df.show(5, truncate=False)


## Part B — Write Aircraft Master to Bronze Delta (MERGE)

In [ ]:
# ── Write Aircraft Master Bronze Delta (MERGE on aircraft_registration)
bronze_aircraft_path = f"{BRONZE_DIR}/aircraft_master"

if DeltaTable.isDeltaTable(spark, bronze_aircraft_path):
    aircraft_table = DeltaTable.forPath(spark, bronze_aircraft_path)
    aircraft_table.alias("tgt").merge(
        raw_aircraft_df.alias("src"),
        "tgt.aircraft_registration = src.aircraft_registration"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed on {bronze_aircraft_path}")
else:
    (raw_aircraft_df.write
        .format("delta")
        .mode("overwrite")
        .save(bronze_aircraft_path))
    print(f"Initial load completed to {bronze_aircraft_path}")

bronze_aircraft_df = spark.read.format("delta").load(bronze_aircraft_path)
print(f"Bronze aircraft_master total rows: {bronze_aircraft_df.count()}")


## Cross-Validation Report

In [ ]:
# ── Cross-Validation ──────────────────────────────────────────────
bronze_flight_df = spark.read.format("delta").load(f"{BRONZE_DIR}/flight_schedule")

# 1. Every airline_code in flight_schedule should have an SLA contract
flight_airlines = bronze_flight_df.select("airline_code").distinct()
sla_airlines = bronze_sla_df.select("airline_code").distinct()
airlines_without_sla = flight_airlines.subtract(sla_airlines)
print(f"Airline codes in flights WITHOUT SLA contract: {airlines_without_sla.count()}")
if airlines_without_sla.count() > 0:
    airlines_without_sla.show()

# 2. Every aircraft_registration in flight_schedule should have master record
flight_regs = bronze_flight_df.select("aircraft_registration").distinct()
master_regs = bronze_aircraft_df.select("aircraft_registration").distinct()
regs_without_master = flight_regs.subtract(master_regs)
print(f"Aircraft registrations WITHOUT master record: {regs_without_master.count()}")
if regs_without_master.count() > 0:
    regs_without_master.show()
else:
    print("\nAll cross-validation checks PASSED.")
